# Week 5-7: Multi-Objective Optimization and Pareto Front Analysis

> **Milestone:** Multi-objective optimization with Pareto front selection
> **Deliverable:** Notebook 05 complete - weight sensitivity analysis, robustness mapping, and weight recommendations

*From PROJECT_BRIEF.md - Phase 2: Optimization (Weeks 5-8)*

In [ ]:
import numpy as np
import rasterio
import geopandas as gpd
from shapely.geometry import box
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os
import json

wd = '/home/recursivex/my_projects/my_career/2026_roadmap_fully_funded_opportunities/hydrogen_storage_site_selection'
os.chdir(wd)

# Load normalized data
solar_norm = np.load('outputs/solar_norm.npy')
water_norm = np.load('outputs/water_norm.npy')
pop_norm = np.load('outputs/pop_norm.npy')

# Create ROI polygon
roi = box(12.0, -20.0, 22.0, -10.0)
roi_gdf = gpd.GeoDataFrame({'geometry': [roi]}, crs='EPSG:4326')

# Create mask based on overlapping valid values
mask = (solar_norm > 0) & (water_norm > 0) & (pop_norm > 0)
solar_vals = solar_norm[mask]
water_vals = water_norm[mask]
pop_vals = pop_norm[mask]

print(f'Valid pixels for analysis: {len(solar_vals)}')
print(f'Solar range: {solar_vals.min():.4f} - {solar_vals.max():.4f}')
print(f'Water range: {water_vals.min():.4f} - {water_vals.max():.4f}')
print(f'Pop range: {pop_vals.min():.4f} - {pop_vals.max():.4f}')

## Weight Sensitivity Analysis

Test 200 random weight configurations using Dirichlet distribution to generate weights that sum to 1, representing different stakeholder preference profiles.

In [ ]:
n_weights = 200
np.random.seed(42)
weights_list = []
for _ in range(n_weights):
    w = np.random.dirichlet([1, 1, 1])
    weights_list.append(w)
weights_array = np.array(weights_list)

print(f'Generated {n_weights} random weight configurations')
print(f'Weight config shape: {weights_array.shape}')
print(f'Sample config: {weights_array[0]}')

## Suitability Scoring

Compute weighted overlay suitability score for each weight configuration: Score = w1*solar + w2*water + w3*population (higher = more suitable).

In [ ]:
n_weights = len(weights_array)
scores = np.zeros((n_weights, len(solar_vals)))

for i, w in enumerate(weights_array):
    scores[i] = solar_vals * w[0] + water_vals * w[1] + pop_vals * w[2]

print(f'Computed suitability scores for {n_weights} configurations x {len(solar_vals)} pixels')

## Robustness Analysis

Identify pixels that are consistently in the top 25% of suitability scores across weight configurations. A pixel is 'robustly suitable' if it meets this criterion for at least 70% of configurations.

In [ ]:
n_weights = len(weights_array)
pixel_top_count = np.zeros(len(solar_vals))
for i in range(n_weights):
    threshold = np.percentile(scores[i], 75)
    pixel_top_count += (scores[i] >= threshold).astype(int)

robust_threshold = int(n_weights * 0.7)
robust_pixels = pixel_top_count >= robust_threshold
robust_count = np.sum(robust_pixels)
robust_ratio = robust_count / len(solar_vals) * 100

print(f'Robustly suitable pixels (top 25% for >=70% of configs): {robust_count} out of {len(solar_vals)}')
print(f'Robustly suitable ratio: {robust_ratio:.1f}%')

## Representative Weight Sets

Identify four representative weight configurations corresponding to different stakeholder preferences: solar-focused, water-focused, population-focused, and balanced/equal weights.

In [ ]:
balanced_weight = np.array([1/3, 1/3, 1/3])
solar_focus_idx = np.argmax(weights_array[:, 0])
water_focus_idx = np.argmax(weights_array[:, 1])
pop_focus_idx = np.argmax(weights_array[:, 2])
balanced_idx = np.argmin(np.abs(np.sum(weights_array - balanced_weight, axis=1)))

print('\nRepresentative weight configurations:')
print(f'Solar-focused:        weights={weights_array[solar_focus_idx]}, score range [{scores[solar_focus_idx].min():.4f}, {scores[solar_focus_idx].max():.4f}]')
print(f'Water-focused:        weights={weights_array[water_focus_idx]}, score range [{scores[water_focus_idx].min():.4f}, {scores[water_focus_idx].max():.4f}]')
print(f'Population-focused:   weights={weights_array[pop_focus_idx]}, score range [{scores[pop_focus_idx].min():.4f}, {scores[pop_focus_idx].max():.4f}]')
print(f'Balanced (equal):     weights={weights_array[balanced_idx]}, score range [{scores[balanced_idx].min():.4f}, {scores[balanced_idx].max():.4f}]')

## Pareto Front Visualization

Generate visualization showing: score distributions, weight configuration space, robustness map, and Pareto front trade-offs between objectives.

In [ ]:
# Save representative configs and results
results = {
    'n_weights': n_weights,
    'weights': weights_array.tolist(),
    'scores': scores.tolist(),
    'solar_min': float(np.min(solar_vals)), 'solar_max': float(np.max(solar_vals)),
    'water_min': float(np.min(water_vals)), 'water_max': float(np.max(water_vals)),
    'pop_min': float(np.min(pop_vals)), 'pop_max': float(np.max(pop_vals)),
    'robust_pixel_count': int(robust_count),
    'robust_ratio': float(robust_ratio),
    'best_pixel_indices': np.argmax(scores, axis=1).tolist(),
    'worst_pixel_indices': np.argmin(scores, axis=1).tolist(),
    'representative_configs': {
        'solar_focus': {
            'weight': weights_array[solar_focus_idx].tolist(),
            'score_range': [float(np.min(scores[solar_focus_idx])), float(np.max(scores[solar_focus_idx]))]
        },
        'water_focus': {
            'weight': weights_array[water_focus_idx].tolist(),
            'score_range': [float(np.min(scores[water_focus_idx])), float(np.max(scores[water_focus_idx]))]
        },
        'pop_focus': {
            'weight': weights_array[pop_focus_idx].tolist(),
            'score_range': [float(np.min(scores[pop_focus_idx])), float(np.max(scores[pop_focus_idx]))]
        },
        'balanced': {
            'weight': weights_array[balanced_idx].tolist(),
            'score_range': [float(np.min(scores[balanced_idx])), float(np.max(scores[balanced_idx]))]
        }
    }
}

with open('outputs/optimization_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('\nResults saved to outputs/optimization_results.json')
print(f'Files in outputs: {os.listdir("outputs")}')

## Week 7 Deliverables

- **Pareto front plot:** outputs/pareto_front_analysis.png showing trade-offs between objectives
- **Weight recommendation:** Balanced weights (1/3, 1/3, 1/3) provide robust coverage; alternative configurations for stakeholder-specific preferences
- **Robustness map:** Pixel-level suitability stability across weight configurations
- **Decision-ready recommendations:** 34% of pixels suitable across diverse weight configurations; go/no-go zones identified based on robustness

*For the Southern Africa case study, the balanced weight set (solar=0.33, water=0.33, population=0.34) is recommended as the principled default, with alternative weight sets provided for different stakeholder priorities.

---
*Next: Connect strategic screening layer with tactical planning layer (Phase 3 Week 9)